### Importing libraries

In [1]:
import pandas as pd
import numpy as np
import re
import wandb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

### Preprocessing

In [4]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

In [5]:
cols = ["prompt","A","B","C","D","E"]

for col in cols:
    train[col] = train[col].apply(clean)
    test[col] = test[col].apply(clean)

In [6]:
labels = ["A","B","C","D","E"]

texts = []
targets = []
question_ids = []

In [7]:
for i in range(len(train)):
    
    prompt = train.loc[i, "prompt"]
    answer = train.loc[i, "answer"]
    
    for label in labels:
        option = train.loc[i, label]
        
        text = prompt + " [SEP] " + option
        texts.append(text)
        
        if label == answer:
            targets.append(1)
        else:
            targets.append(0)
            
        question_ids.append(i)

In [8]:
print(len(texts))

10000


### Split p=on basis of train and validation

In [9]:
question_indices = list(range(len(train)))

train_q, val_q = train_test_split(
    question_indices,
    test_size=0.2,
    random_state=42
)

In [10]:
train_texts = []
train_targets = []

val_texts = []
val_targets = []

In [11]:
for idx, qid in enumerate(question_ids):
    
    if qid in train_q:
        train_texts.append(texts[idx])
        train_targets.append(targets[idx])
    else:
        val_texts.append(texts[idx])
        val_targets.append(targets[idx])

In [12]:
print(len(train_texts))
print(len(train_targets))

print(len(val_texts))
print(len(val_targets))

8000
8000
2000
2000


### Tokenization

In [13]:
MAX_LEN = 128

word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for text in train_texts:
    for word in text.split():
        if word not in word2idx:
            word2idx[word] = len(word2idx)

vocab = word2idx

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 3090


In [14]:
def text_to_sequence(text, vocab, max_len=128):

    sequence = []

    for word in text.split():
        sequence.append(
            vocab.get(word, vocab["<UNK>"])
        )

    if len(sequence) > max_len:
        sequence = sequence[:max_len]

    else:
        sequence += [vocab["<PAD>"]] * (max_len - len(sequence))

    return sequence

In [15]:
import torch
from torch.utils.data import Dataset

class MCQDataset(Dataset):

    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return {
            "input_ids": torch.tensor(
                self.sequences[idx],
                dtype=torch.long
            ),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.float
            )
        }

In [16]:
train_sequences = [
    text_to_sequence(text, vocab, MAX_LEN)
    for text in train_texts
]

val_sequences = [
    text_to_sequence(text, vocab, MAX_LEN)
    for text in val_texts
]

In [17]:
train_dataset = MCQDataset(
    train_sequences,
    train_targets
)

val_dataset = MCQDataset(
    val_sequences,
    val_targets
)

### Dataloader

In [18]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [19]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["labels"].shape)

torch.Size([64, 128])
torch.Size([64])


In [20]:
import torch.nn as nn

### BiLSTM + Attention module blocks

In [21]:
class BiLSTMAttention(nn.Module):

    def __init__(self,
                 vocab_size,
                 embed_dim=128,
                 hidden_dim=128):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.attention = nn.Linear(
            hidden_dim * 2,
            1
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(
            hidden_dim * 2,
            1
        )

    def forward(self, input_ids):

        embedded = self.embedding(input_ids)

        lstm_output, _ = self.lstm(embedded)

        attention_scores = self.attention(
            lstm_output
        ).squeeze(-1)

        attention_weights = torch.softmax(
            attention_scores,
            dim=1
        )

        context = torch.sum(
            lstm_output *
            attention_weights.unsqueeze(-1),
            dim=1
        )

        context = self.dropout(context)

        output = self.fc(context)

        return output.squeeze()

In [22]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = BiLSTMAttention(
    vocab_size=len(vocab)
).to(device)

In [23]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [24]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb")

wandb.login(key=wandb_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: taransaggu (taransaggu-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [25]:
run = wandb.init(
    project="DL-GenAI-2026-T2",
    name="bilstm_attention_scratch_final",
    config={
        "model": "BiLSTM + Attention",
        "epochs": 10,
        "embedding_dim": 128,
        "hidden_dim": 128,
        "optimizer": "Adam",
        "learning_rate": 1e-3,
        "loss": "BCEWithLogitsLoss"
    }
)

### tarining

In [26]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    avg_loss = train_loss / len(train_loader)

    print(
        f"Epoch {epoch+1} "
        f"Loss: {avg_loss:.4f}"
    )

    run.log({
        "epoch": epoch + 1,
        "train_loss": avg_loss
    })

Epoch 1 Loss: 0.5113
Epoch 2 Loss: 0.4551
Epoch 3 Loss: 0.3194
Epoch 4 Loss: 0.2128
Epoch 5 Loss: 0.1266
Epoch 6 Loss: 0.0922
Epoch 7 Loss: 0.0574
Epoch 8 Loss: 0.0437
Epoch 9 Loss: 0.0371
Epoch 10 Loss: 0.0199


In [27]:
model.eval()

val_loss = 0

correct = 0
total = 0

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids)

        loss = criterion(outputs, labels)
        val_loss += loss.item()

        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

avg_val_loss = val_loss / len(val_loader)
val_accuracy = correct / total

print(f"Validation Loss: {avg_val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

run.log({
    "validation_loss": avg_val_loss,
    "validation_accuracy": val_accuracy,
})

Validation Loss: 0.0238
Validation Accuracy: 0.9915


In [28]:
model.eval()

val_probs = []

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch["input_ids"].to(device)

        outputs = model(input_ids)

        probs = torch.sigmoid(outputs)

        val_probs.extend(
            probs.cpu().numpy()
        )

In [29]:
print(len(val_probs))

2000


In [30]:
val_question_ids = []

for idx, qid in enumerate(question_ids):

    if qid not in train_q:
        val_question_ids.append(qid)

print(len(val_question_ids))

2000


In [31]:
val_answers = []

for idx, qid in enumerate(question_ids):

    if qid not in train_q:
        val_answers.append(targets[idx])

In [32]:
import numpy as np

def apk(actual, predicted, k=3):

    predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):

        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score

In [33]:
letters = ["A", "B", "C", "D", "E"]

scores = []

for i in range(0, len(val_probs), 5):

    probs = val_probs[i:i+5]

    labels = val_answers[i:i+5]

    correct_index = labels.index(1)

    actual = letters[correct_index]

    ranked = np.argsort(probs)[::-1]

    predicted = [
        letters[idx]
        for idx in ranked[:3]
    ]

    scores.append(
        apk(actual, predicted)
    )

map3 = np.mean(scores)

print(f"Validation MAP@3 : {map3:.4f}")

Validation MAP@3 : 0.9946


In [34]:
run.log({
    "validation_map3": map3
})

### Test prediction

In [35]:
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

test_texts = []
test_ids = []

for _, row in test_df.iterrows():

    for option in ["A", "B", "C", "D", "E"]:

        text = (
            row["prompt"]
            + " [SEP] "
            + row[option]
        )

        test_texts.append(text)
        test_ids.append(row["id"])

print(len(test_texts))

2500


In [36]:
test_sequences = [
    text_to_sequence(text, vocab, MAX_LEN)
    for text in test_texts
]

In [37]:
class TestDataset(Dataset):

    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):

        return {
            "input_ids": torch.tensor(
                self.sequences[idx],
                dtype=torch.long
            )
        }

In [38]:
test_dataset = TestDataset(
    test_sequences
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [39]:
model.eval()

test_probs = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)

        outputs = model(input_ids)

        probs = torch.sigmoid(outputs)

        test_probs.extend(
            probs.cpu().numpy()
        )

print(len(test_probs))

2500


In [40]:
letters = ["A", "B", "C", "D", "E"]

predictions = []

for i in range(0, len(test_probs), 5):

    probs = test_probs[i:i+5]

    ranked = np.argsort(probs)[::-1]

    top3 = " ".join(
        letters[idx]
        for idx in ranked[:3]
    )

    predictions.append(top3)

In [41]:
len(predictions)

500

### Submission.csv file

In [42]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

torch.save(
    model.state_dict(),
    "bilstm_attention.pt"
)

run.finish()

epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▇▅▄▃▂▂▁▁▁
validation_accuracy,▁
validation_loss,▁
validation_map3,▁
epoch,10
train_loss,0.01989
validation_accuracy,0.9915
validation_loss,0.02385
validation_map3,0.99458
